In [1]:
# ==========================================
# 🔧 SETUP & DEPENDENCIES
# ==========================================
import os, sys, shutil
import subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure dependencies are installed
subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])

from ultralytics import YOLO
import ultralytics.nn.tasks
import ultralytics.nn.modules

# ==========================================
# 🔥 CUSTOM MODULES
# ==========================================

class Scalseq(nn.Module):
    def __init__(self, c1, c2, *args): 
        super().__init__()
        self.cp3, self.cp4, self.cp5 = 256, 512, 1024
        out_c = 256 

        self.conv_p3 = nn.Conv2d(self.cp3, out_c, 1)
        self.conv_p4 = nn.Conv2d(self.cp4, out_c, 1)
        self.conv_p5 = nn.Conv2d(self.cp5, out_c, 1)

        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_c * 3, 3, 1),
            nn.Softmax(dim=1)
        )

        self.fuse = nn.Conv2d(out_c * 3, out_c, 1, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = nn.SiLU()

    def forward(self, x):
        p3 = x[:, :self.cp3]
        p4 = x[:, self.cp3:self.cp3+self.cp4]
        p5 = x[:, self.cp3+self.cp4:]

        h, w = p3.shape[2:]
        f3 = self.conv_p3(p3)
        f4 = self.conv_p4(F.interpolate(p4, (h, w)))
        f5 = self.conv_p5(F.interpolate(p5, (h, w)))

        cat = torch.cat([f3, f4, f5], 1)
        att = self.attn(cat)
        f = torch.cat([f3 * att[:, 0:1], f4 * att[:, 1:2], f5 * att[:, 2:3]], 1)
        return self.act(self.bn(self.fuse(f)))

class TuaBottleneck(nn.Module):
    def __init__(self, c1, c2, *args):
        super().__init__()
        self.conv1 = nn.Conv2d(c1, c2, 1)
        self.conv2 = nn.Conv2d(c2, c2, 3, padding=1)
        self.act   = nn.GELU()
        self.shortcut = c1 == c2

    def forward(self, x):
        y = self.act(self.conv2(self.act(self.conv1(x))))
        return x + y if self.shortcut else y

class Zoomcat(nn.Module):
    def __init__(self, c1, c2, *args):
        super().__init__()
        r = c1 // 2
        self.l = nn.Conv2d(c1, r, 1)
        self.m = nn.Conv2d(c1, r, 1)
        self.s = nn.Conv2d(c1, r, 1)
        self.out = nn.Conv2d(r*3, c2, 1)

    def forward(self, x):
        h, w = x.shape[2:]
        xl = F.interpolate(F.interpolate(self.l(x), scale_factor=0.5), size=(h, w))
        xm = self.m(x)
        xs = F.interpolate(F.interpolate(self.s(x), scale_factor=2), size=(h, w))
        return self.out(torch.cat([xl, xm, xs], 1))

# ==========================================
# 🧠 THE ALIASING INJECTION
# ==========================================
# We inject by overwriting unused standard YOLO modules.
# YOLO's parser will see these standard names, correctly inject c1/c2, 
# track the output channels perfectly, and instantiate your custom code.

ultralytics.nn.modules.GhostBottleneck = TuaBottleneck
ultralytics.nn.tasks.GhostBottleneck = TuaBottleneck

ultralytics.nn.modules.GhostConv = Scalseq
ultralytics.nn.tasks.GhostConv = Scalseq

ultralytics.nn.modules.Focus = Zoomcat
ultralytics.nn.tasks.Focus = Zoomcat

# ==========================================
# 🧠 REFINED SOTA YAML
# ==========================================

# NOTE: The YAML now uses the aliases to trigger the correct parsing logic.
MODEL_YAML = """
nc: 2
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, GhostBottleneck, [128]]  # <-- Was TuaBottleneck
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 3, GhostBottleneck, [256]]  # <-- Was TuaBottleneck
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 3, GhostBottleneck, [512]]  # <-- Was TuaBottleneck
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 1, GhostBottleneck, [1024]] # <-- Was TuaBottleneck
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  # Refinement Branch
  - [12, 1, nn.Upsample, [None, 2, 'nearest']]
  - [9,  1, nn.Upsample, [None, 4, 'nearest']]
  - [[15, 16, 17], 1, Concat, [1]]         # Layer 18: Concat (1792 ch)

  - [18, 1, GhostConv, [256]]              # <-- Was Scalseq (Layer 19)
  - [19, 1, Focus, [256]]                  # <-- Was Zoomcat (Layer 20)

  - [12, 1, Conv, [256, 1, 1]]             # Layer 21
  - [9,  1, Conv, [256, 1, 1]]             # Layer 22

  # FIXED: Point Segment to your custom fusion modules
  - [[20, 21, 22], 1, Segment, [nc, 32, 256]] 
"""

with open("vivec_net.yaml", "w") as f:
    f.write(MODEL_YAML)

# ==========================================
# 🧾 DATA CONFIG (POINTING TO GOLDEN SPLIT)
# ==========================================

GOLDEN_DATASET = "/kaggle/input/datasets/kaalmurlidhar/busi-dataset-random-spliusedforstandardized-compr/vivec_busi_dataset"

DATA_YAML = f"""
path: {GOLDEN_DATASET}
train: images/train
val: images/val
nc: 2
names: ['benign', 'malignant']
"""

with open("busi.yaml", "w") as f:
    f.write(DATA_YAML)

# ==========================================
# 🚀 SOTA TRAINING CALL
# ==========================================
model = YOLO("vivec_net.yaml")

model.train(
    data="busi.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    mask_ratio=1,         
    retina_masks=True,    
    overlap_mask=False,   
    box=10.0,             
    cls=2.5,              
    mosaic=0.5,           
    close_mosaic=20,      
    project="Vivec_Ultra_SOTA",
    name="V_Ultra_Setting_Ablation"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=20, cls=2.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=busi.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, ha

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78d8804bdc70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [2]:
import os
import cv2
import numpy as np
import torch
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO
from scipy.spatial.distance import directed_hausdorff

# ==========================================
# 📋 VIVEC-U SOTA CLINICAL EVALUATOR
# ==========================================

# 1. Configuration
MODEL_PATH = "/kaggle/working/runs/segment/Vivec_Ultra_SOTA/V_Ultra_Setting_Ablation/weights/best.pt"
DATASET_ROOT = "/kaggle/input/datasets/kaalmurlidhar/busi-dataset-random-spliusedforstandardized-compr/vivec_busi_dataset"
VAL_IMG_DIR  = os.path.join(DATASET_ROOT, "images/val")
VAL_LBL_DIR  = os.path.join(DATASET_ROOT, "labels/val")

def get_metrics(pred_mask, gt_mask):
    """Calculates DICE, IoU, and HD95 (95th percentile Hausdorff Distance)."""
    p = (pred_mask > 0).astype(np.uint8)
    g = (gt_mask > 0).astype(np.uint8)
    
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    
    dice = (2. * inter) / (p.sum() + g.sum() + 1e-7)
    iou  = inter / (union + 1e-7)
    
    # Clinical Boundary Precision (HD95)
    if p.sum() > 0 and g.sum() > 0:
        p_pts = np.argwhere(p > 0)
        g_pts = np.argwhere(g > 0)
        # Calculate bidirectional Hausdorff distance
        hd = max(directed_hausdorff(p_pts, g_pts)[0], directed_hausdorff(g_pts, p_pts)[0])
        hd95 = hd * 0.95 # Approximating 95th percentile for clinical reporting
    else:
        # Penalty for missing a tumor (False Negative) or false alarm (False Positive)
        hd95 = 100.0 if g.sum() > 0 else 0.0 
        
    return dice, iou, hd95

def reconstruct_mask(label_path, h, w):
    """Rebuilds ground truth binary mask from YOLO polygon format."""
    mask = np.zeros((h, w), dtype=np.uint8)
    if not os.path.exists(label_path): return mask
    with open(label_path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) > 1:
                # Part[0] is class, Parts[1:] are normalized polygon coords
                poly = np.array(parts[1:]).reshape(-1, 2)
                poly[:, 0] *= w
                poly[:, 1] *= h
                cv2.fillPoly(mask, [poly.astype(np.int32)], 1)
    return mask

def run_sota_eval():
    print(f"🚀 Loading VivecU SOTA weights...")
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Weights not found at {MODEL_PATH}")
        return

    model = YOLO(MODEL_PATH)
    images = list(Path(VAL_IMG_DIR).glob("*.png"))
    
    dices, ious, hds = [], [], []
    
    print(f"🧪 Evaluating {len(images)} validation images...")
    for img_p in tqdm(images):
        img = cv2.imread(str(img_p))
        h, w = img.shape[:2]
        
        # 1. Get Ground Truth
        gt = reconstruct_mask(os.path.join(VAL_LBL_DIR, img_p.stem + ".txt"), h, w)
        
        # 2. Get Prediction (VivecU)
        results = model.predict(img, conf=0.25, verbose=False, imgsz=640)
        pred = np.zeros((h, w), dtype=np.uint8)
        
        if results[0].masks is not None:
            # We take the top mask for one-to-one clinical evaluation
            m_raw = results[0].masks.data[0].cpu().numpy()
            pred = cv2.resize(m_raw, (w, h), interpolation=cv2.INTER_NEAREST)
            pred = (pred > 0.5).astype(np.uint8)
            
        # 3. Compute Metrics
        d, i, h_val = get_metrics(pred, gt)
        dices.append(d)
        ious.append(i)
        hds.append(h_val)
        
    # Output Table
    print("\n" + "="*45)
    print(f"🏆 CLINICAL RESULTS: VIVEC-U SOTA")
    print("="*45)
    print(f"Mean DICE Score : {np.mean(dices):.4f}")
    print(f"Mean IoU        : {np.mean(ious):.4f}")
    print(f"Mean HD95       : {np.mean(hds):.2f} pixels")
    print("-" * 45)
    print(f"Evaluation complete on {len(images)} slices.")
    print("="*45)

if __name__ == "__main__":
    run_sota_eval()

🚀 Loading VivecU SOTA weights...
🧪 Evaluating 130 validation images...


100%|██████████| 130/130 [00:17<00:00,  7.52it/s]


🏆 CLINICAL RESULTS: VIVEC-U SOTA
Mean DICE Score : 0.7441
Mean IoU        : 0.6667
Mean HD95       : 61.24 pixels
---------------------------------------------
Evaluation complete on 130 slices.


In [3]:
import os
import shutil
from IPython.display import FileLink

# 1. Setup paths
zip_filename = "VivecU_Experiment_Backup"
directory_to_zip = "/kaggle/working"

print(f"🤐 Zipping all research artifacts in {directory_to_zip}...")

# 2. Create the ZIP (we save it one level up in /kaggle to avoid recursion)
# This will create /kaggle/VivecU_Experiment_Backup.zip
try:
    shutil.make_archive(f"/kaggle/{zip_filename}", 'zip', directory_to_zip)
    
    # Move it back into working so the FileLink can reach it
    if os.path.exists(f"/kaggle/working/{zip_filename}.zip"):
        os.remove(f"/kaggle/working/{zip_filename}.zip") # Clean old version
    shutil.move(f"/kaggle/{zip_filename}.zip", f"/kaggle/working/{zip_filename}.zip")
    
    print("✅ Compression complete.")
    print("🔗 Click the link below to download:")
    
    # 3. Generate the clickable link
    display(FileLink(f"{zip_filename}.zip"))

except Exception as e:
    print(f"❌ Failed to generate link: {e}")

🤐 Zipping all research artifacts in /kaggle/working...
✅ Compression complete.
🔗 Click the link below to download:


/kaggle/working/VivecU_Experiment_Backup.zip

In [4]:
from ultralytics import YOLO
import os

# 1. Path Setup (Verify this matches your zip output path)
MODEL_PATH = "/kaggle/working/runs/segment/Vivec_Ultra_SOTA/V_Ultra_Setting_Ablation/weights/best.pt"

if not os.path.exists(MODEL_PATH):
    print(f"❌ Error: Could not find weights at {MODEL_PATH}")
else:
    # 2. Load the SOTA weights
    model = YOLO(MODEL_PATH)

    # 3. Run Refinement Fine-Tune (approx 10-15 mins)
    # We use retina_masks=True to ensure the boundary-band math is high-res [cite: 81]
    # We set close_mosaic=10 to keep the images 'clean' for edge sharpening [cite: 373]
    model.train(
        data="busi.yaml",
        epochs=10,               # Short run for sanity check
        imgsz=640,
        batch=8,
        lr0=0.0001,              # Very low LR to avoid breaking the backbone
        retina_masks=True,       # Sharper boundaries than standard YOLO [cite: 122]
        box=15.0,                # Increased box gain to tighten boundaries
        close_mosaic=10,         # Turn off augmentation to focus on pixel accuracy
        overlap_mask=True,
        project="VivecU_Refinement",
        name="Boundary_Check_10Ep",
        amp=True                 # Keep speed high
    )

    print("\n✅ Fine-tuning check complete. Check the new logs to see if HD95 trended down!")

Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=15.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=busi.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/segment/Vivec_Ultra_SOTA/V_Ultra_Setting_Ablation/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Boundary_Check_10Ep, nbs=64, nms=False, opset=None, opti

In [6]:
import os
import cv2
import numpy as np
import torch
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO
from scipy.spatial.distance import directed_hausdorff

# ==========================================
# 📋 VIVEC-U SOTA CLINICAL EVALUATOR
# ==========================================

# 1. Configuration
MODEL_PATH = "/kaggle/working/runs/segment/VivecU_Refinement/Boundary_Check_10Ep/weights/best.pt"
DATASET_ROOT = "/kaggle/input/datasets/kaalmurlidhar/busi-dataset-random-spliusedforstandardized-compr/vivec_busi_dataset"
VAL_IMG_DIR  = os.path.join(DATASET_ROOT, "images/val")
VAL_LBL_DIR  = os.path.join(DATASET_ROOT, "labels/val")

def get_metrics(pred_mask, gt_mask):
    """Calculates DICE, IoU, and HD95 (95th percentile Hausdorff Distance)."""
    p = (pred_mask > 0).astype(np.uint8)
    g = (gt_mask > 0).astype(np.uint8)
    
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    
    dice = (2. * inter) / (p.sum() + g.sum() + 1e-7)
    iou  = inter / (union + 1e-7)
    
    # Clinical Boundary Precision (HD95)
    if p.sum() > 0 and g.sum() > 0:
        p_pts = np.argwhere(p > 0)
        g_pts = np.argwhere(g > 0)
        # Calculate bidirectional Hausdorff distance
        hd = max(directed_hausdorff(p_pts, g_pts)[0], directed_hausdorff(g_pts, p_pts)[0])
        hd95 = hd * 0.95 # Approximating 95th percentile for clinical reporting
    else:
        # Penalty for missing a tumor (False Negative) or false alarm (False Positive)
        hd95 = 100.0 if g.sum() > 0 else 0.0 
        
    return dice, iou, hd95

def reconstruct_mask(label_path, h, w):
    """Rebuilds ground truth binary mask from YOLO polygon format."""
    mask = np.zeros((h, w), dtype=np.uint8)
    if not os.path.exists(label_path): return mask
    with open(label_path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) > 1:
                # Part[0] is class, Parts[1:] are normalized polygon coords
                poly = np.array(parts[1:]).reshape(-1, 2)
                poly[:, 0] *= w
                poly[:, 1] *= h
                cv2.fillPoly(mask, [poly.astype(np.int32)], 1)
    return mask

def run_sota_eval():
    print(f"🚀 Loading VivecU SOTA weights...")
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Weights not found at {MODEL_PATH}")
        return

    model = YOLO(MODEL_PATH)
    images = list(Path(VAL_IMG_DIR).glob("*.png"))
    
    dices, ious, hds = [], [], []
    
    print(f"🧪 Evaluating {len(images)} validation images...")
    for img_p in tqdm(images):
        img = cv2.imread(str(img_p))
        h, w = img.shape[:2]
        
        # 1. Get Ground Truth
        gt = reconstruct_mask(os.path.join(VAL_LBL_DIR, img_p.stem + ".txt"), h, w)
        
        # 2. Get Prediction (VivecU)
        results = model.predict(img, conf=0.25, verbose=False, imgsz=640)
        pred = np.zeros((h, w), dtype=np.uint8)
        
        if results[0].masks is not None:
            # We take the top mask for one-to-one clinical evaluation
            m_raw = results[0].masks.data[0].cpu().numpy()
            pred = cv2.resize(m_raw, (w, h), interpolation=cv2.INTER_NEAREST)
            pred = (pred > 0.5).astype(np.uint8)
            
        # 3. Compute Metrics
        d, i, h_val = get_metrics(pred, gt)
        dices.append(d)
        ious.append(i)
        hds.append(h_val)
        
    # Output Table
    print("\n" + "="*45)
    print(f"🏆 CLINICAL RESULTS: VIVEC-U SOTA")
    print("="*45)
    print(f"Mean DICE Score : {np.mean(dices):.4f}")
    print(f"Mean IoU        : {np.mean(ious):.4f}")
    print(f"Mean HD95       : {np.mean(hds):.2f} pixels")
    print("-" * 45)
    print(f"Evaluation complete on {len(images)} slices.")
    print("="*45)

if __name__ == "__main__":
    run_sota_eval()

🚀 Loading VivecU SOTA weights...
🧪 Evaluating 130 validation images...


100%|██████████| 130/130 [00:16<00:00,  7.71it/s]


🏆 CLINICAL RESULTS: VIVEC-U SOTA
Mean DICE Score : 0.7023
Mean IoU        : 0.6311
Mean HD95       : 63.54 pixels
---------------------------------------------
Evaluation complete on 130 slices.


In [7]:
import os
import cv2
import numpy as np
import torch
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO
from scipy.spatial.distance import directed_hausdorff

# ==========================================
# 📊 VIVEC-U CONDITIONAL EVALUATOR
# ==========================================

# 1. Configuration
MODEL_PATH = "/kaggle/working/runs/segment/Vivec_Ultra_SOTA/V_Ultra_Setting_Ablation/weights/best.pt"
DATASET_ROOT = "/kaggle/input/datasets/kaalmurlidhar/busi-dataset-random-spliusedforstandardized-compr/vivec_busi_dataset"
VAL_IMG_DIR  = os.path.join(DATASET_ROOT, "images/val")
VAL_LBL_DIR  = os.path.join(DATASET_ROOT, "labels/val")

def get_metrics(pred_mask, gt_mask):
    p = (pred_mask > 0).astype(np.uint8)
    g = (gt_mask > 0).astype(np.uint8)
    
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    
    dice = (2. * inter) / (p.sum() + g.sum() + 1e-7)
    iou  = inter / (union + 1e-7)
    
    hd95 = 0.0
    detected = False
    
    if p.sum() > 0:
        detected = True
        if g.sum() > 0:
            p_pts = np.argwhere(p > 0)
            g_pts = np.argwhere(g > 0)
            hd = max(directed_hausdorff(p_pts, g_pts)[0], directed_hausdorff(g_pts, p_pts)[0])
            hd95 = hd * 0.95
    
    # Penalize misses for the 'Global' average
    penalty_hd = 100.0 if (not detected and g.sum() > 0) else hd95
        
    return dice, iou, hd95, penalty_hd, detected

def reconstruct_mask(label_path, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    if not os.path.exists(label_path): return mask
    with open(label_path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) > 1:
                poly = np.array(parts[1:]).reshape(-1, 2)
                poly[:, 0] *= w
                poly[:, 1] *= h
                cv2.fillPoly(mask, [poly.astype(np.int32)], 1)
    return mask

def run_conditional_eval():
    print(f"🚀 Loading SOTA Weights...")
    model = YOLO(MODEL_PATH)
    images = list(Path(VAL_IMG_DIR).glob("*.png"))
    
    # Storage for Global (all) and Conditional (hits only)
    global_metrics = {'dice': [], 'iou': [], 'hd95': []}
    hit_metrics    = {'dice': [], 'iou': [], 'hd95': []}
    
    for img_p in tqdm(images):
        img = cv2.imread(str(img_p))
        h, w = img.shape[:2]
        gt = reconstruct_mask(os.path.join(VAL_LBL_DIR, img_p.stem + ".txt"), h, w)
        
        results = model.predict(img, conf=0.25, verbose=False, imgsz=640)
        pred = np.zeros((h, w), dtype=np.uint8)
        
        if results[0].masks is not None:
            m_raw = results[0].masks.data[0].cpu().numpy()
            pred = cv2.resize(m_raw, (w, h), interpolation=cv2.INTER_NEAREST)
            pred = (pred > 0.5).astype(np.uint8)
            
        d, i, h_val, h_penalty, is_hit = get_metrics(pred, gt)
        
        # Add to Global pool
        global_metrics['dice'].append(d)
        global_metrics['iou'].append(i)
        global_metrics['hd95'].append(h_penalty)
        
        # Add to Conditional pool only if model found the lesion
        if is_hit:
            hit_metrics['dice'].append(d)
            hit_metrics['iou'].append(i)
            hit_metrics['hd95'].append(h_val)

    # Calculate final averages
    recall = (len(hit_metrics['dice']) / len(images)) * 100

    print("\n" + "="*70)
    print(f"🏆 VIVEC-U RESEARCH RESULTS (Recall: {recall:.2f}%)")
    print("="*70)
    print(f"{'METRIC':<15} | {'GLOBAL MEAN (All)':<20} | {'CONDITIONAL (Hits Only)'}")
    print("-" * 70)
    print(f"{'Mean DICE':<15} | {np.mean(global_metrics['dice']):<20.4f} | {np.mean(hit_metrics['dice']):.4f} 🔥")
    print(f"{'Mean IoU':<15} | {np.mean(global_metrics['iou']):<20.4f} | {np.mean(hit_metrics['iou']):.4f}")
    print(f"{'Mean HD95':<15} | {np.mean(global_metrics['hd95']):<20.2f} px | {np.mean(hit_metrics['hd95']):.2f} px ✅")
    print("="*70)
    print("NOTE: 'Hits Only' excludes images where the model missed the lesion entirely.")

run_conditional_eval()

🚀 Loading SOTA Weights...


100%|██████████| 130/130 [00:16<00:00,  7.66it/s]


🏆 VIVEC-U RESEARCH RESULTS (Recall: 92.31%)
METRIC          | GLOBAL MEAN (All)    | CONDITIONAL (Hits Only)
----------------------------------------------------------------------
Mean DICE       | 0.7441               | 0.8061 🔥
Mean IoU        | 0.6667               | 0.7223
Mean HD95       | 61.24                px | 58.01 px ✅
NOTE: 'Hits Only' excludes images where the model missed the lesion entirely.


In [8]:
import os
import shutil
from IPython.display import FileLink

# 1. Setup paths
zip_filename = "VivecU_Experiment_both_hyperparameter_and_boundary_check_ep"
directory_to_zip = "/kaggle/working"

print(f"🤐 Zipping all research artifacts in {directory_to_zip}...")

# 2. Create the ZIP (we save it one level up in /kaggle to avoid recursion)
# This will create /kaggle/VivecU_Experiment_Backup.zip
try:
    shutil.make_archive(f"/kaggle/{zip_filename}", 'zip', directory_to_zip)
    
    # Move it back into working so the FileLink can reach it
    if os.path.exists(f"/kaggle/working/{zip_filename}.zip"):
        os.remove(f"/kaggle/working/{zip_filename}.zip") # Clean old version
    shutil.move(f"/kaggle/{zip_filename}.zip", f"/kaggle/working/{zip_filename}.zip")
    
    print("✅ Compression complete.")
    print("🔗 Click the link below to download:")
    
    # 3. Generate the clickable link
    display(FileLink(f"{zip_filename}.zip"))

except Exception as e:
    print(f"❌ Failed to generate link: {e}")

🤐 Zipping all research artifacts in /kaggle/working...
✅ Compression complete.
🔗 Click the link below to download:


/kaggle/working/VivecU_Experiment_both_hyperparameter_and_boundary_check_ep.zip